# 11. Выбор confidence threshold для Span NER

Порог выбирается **только на validation** по максимальному strict entity-level micro-F1. Модель выполняет один inference-проход; все пороги проверяются по уже рассчитанным вероятностям. Test split здесь не используется.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import runpy

PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
EXPERIMENT_CONFIG = PROJECT_DIR / 'configs/experiments/span_ner_corrected_v1.yaml'
OUTPUT_DIR = PROJECT_DIR / 'results/span_ner_corrected_v1/seed_42'
CHECKPOINT = OUTPUT_DIR / 'checkpoints/best'
BOOTSTRAP = PROJECT_DIR / 'colab_bootstrap.py'

for required_path in (EXPERIMENT_CONFIG, CHECKPOINT, BOOTSTRAP):
    if not required_path.exists():
        raise FileNotFoundError(
            f'Не найден {required_path}. Сначала выполните 10_train_span_ner.ipynb и экспорт checkpoint.'
        )

bootstrap_project = runpy.run_path(str(BOOTSTRAP))['bootstrap_project']
bootstrap_project(PROJECT_DIR)

In [ ]:
import torch

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Предупреждение: calibration выполнится на CPU, но будет заметно медленнее.')

In [ ]:
from rurebus_ie.training import calibrate_span_threshold_experiment

# Текущий threshold=0.50 даёт избыточный recall, поэтому проверяем
# плотную сетку от 0.30 до 0.90 включительно.
THRESHOLDS = [round(0.30 + step * 0.01, 2) for step in range(61)]

calibration = calibrate_span_threshold_experiment(
    EXPERIMENT_CONFIG,
    thresholds=THRESHOLDS,
    project_root=PROJECT_DIR,
    checkpoint_dir=CHECKPOINT,
)
print(f'Оптимальный threshold: {calibration.best_threshold:.2f}')
print(f'Validation micro-F1: {calibration.best_metrics.micro_f1:.4f}')
print(f'Precision: {calibration.best_metrics.precision:.4f}')
print(f'Recall: {calibration.best_metrics.recall:.4f}')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

threshold_table = pd.DataFrame(calibration.rows)
best_index = threshold_table['micro_f1'].idxmax()
display(threshold_table.iloc[max(0, best_index - 5):best_index + 6])

ax = threshold_table.plot(
    x='threshold',
    y=['precision', 'recall', 'micro_f1', 'macro_f1'],
    figsize=(11, 6),
    grid=True,
)
ax.axvline(calibration.best_threshold, color='black', linestyle='--', alpha=0.7)
ax.set_ylim(0, 1)
ax.set_title('Span NER: выбор confidence threshold на validation')
plt.show()

In [ ]:
per_class = pd.DataFrame(calibration.best_metrics.per_class).T.sort_values('f1')
display(per_class)
print('Отчёт:', OUTPUT_DIR / 'threshold_calibration.json')
print('Таблица:', OUTPUT_DIR / 'threshold_calibration.csv')
print('Теперь запускайте 12_test_span_ner.ipynb: он автоматически прочитает выбранный threshold.')